# Production Line Simulator → MQTT (view in MQTT Explorer)
**Certificate 05 · Smart Factory Systems**  |  [← Back to the Lab Hub](../../index.ipynb)

This lab runs a **simulated production line** and publishes its live data to an **MQTT broker**. Open **MQTT Explorer** on your computer, connect to the same broker, and watch the plant's data appear as a live topic tree — the way SCADA, MES, quality (QMS) and ERP systems each subscribe to the slice they care about.

> This is the central scenario for the course's data-integration story. An **OPC UA** version of the same line is a natural companion (compare pub/sub MQTT vs structured OPC UA).

## Objectives
By the end you will be able to:
- Run a production-line simulator that emits realistic, in-spec data.
- Publish it to MQTT as a structured **topic tree**.
- View the live tree in **MQTT Explorer**.
- Explain which plant system (SCADA / MES / QMS / ERP) absorbs which topics.

## Before you start — open MQTT Explorer
1. Install **MQTT Explorer** (free): http://mqtt-explorer.com
2. New connection → **Host** `broker.hivemq.com`, **Port** `1883`, no username/password → **Connect**.
3. Leave it open. Once the simulator below is running, you will see your topics appear.

_`broker.hivemq.com` is a free public test broker — perfect for learning, never for real plant data. Everyone on it shares the namespace, so pick a **unique** student name below._

## Setup

In [ ]:
!pip -q install paho-mqtt

In [ ]:
import json, time, random
import paho.mqtt.client as mqtt

BROKER  = "broker.hivemq.com"
PORT    = 1883
STUDENT = "demo"                 # <-- CHANGE to something unique (your name/initials)
ROOT    = f"coefam/{STUDENT}"

# paho-mqtt 2.x needs a callback API version; 1.x does not — support both
try:
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=f"sim-{STUDENT}-{random.randint(0,9999)}")
except (AttributeError, TypeError):
    client = mqtt.Client(client_id=f"sim-{STUDENT}-{random.randint(0,9999)}")

client.connect(BROKER, PORT, keepalive=60)
client.loop_start()
print("Connected. In MQTT Explorer subscribe to:", ROOT + "/#")

## The production line
A single filling line running a work order. Each second we update the **sensors**, occasionally **complete a part** (~96% pass — an "acceptable" line), and recompute the **KPIs**.

In [ ]:
STATE = dict(produced=0, good=0, reject=0, target=500,
             order="WO-10432", product="Bottle-500ml", t0=time.time())
IDEAL_CYCLE_S = 2.0
REASONS = ["Underweight", "Overweight", "Cap Missing", "Label Skew"]

def tick(k):
    s = STATE
    # continuous, in-spec sensor values
    weight = round(random.gauss(500, 1.2), 1)       # g  (target 500)
    temp   = round(random.gauss(62, 0.8), 1)        # deg C fill temperature
    vib    = round(abs(random.gauss(1.8, 0.25)), 2) # mm/s
    # complete a part roughly every other tick
    result, reason = "", ""
    if k % 2 == 0 and s["produced"] < s["target"]:
        s["produced"] += 1
        if random.random() > 0.04:                  # ~96% good
            s["good"] += 1; result = "Pass"
        else:
            s["reject"] += 1; result = "Fail"; reason = random.choice(REASONS)
    # KPIs (OEE = Availability x Performance x Quality)
    elapsed = max(time.time() - s["t0"], 1)
    availability = 0.98
    performance  = min(1.0, (s["produced"] * IDEAL_CYCLE_S) / elapsed)
    quality      = (s["good"] / s["produced"]) if s["produced"] else 1.0
    oee          = availability * performance * quality
    return dict(weight=weight, temp=temp, vib=vib, result=result, reason=reason,
                availability=round(availability, 3), performance=round(performance, 3),
                quality=round(quality, 3), oee=round(oee, 3))

# quick dry-run of the model (no MQTT) so you can see the data shape
for k in range(1, 4):
    print(tick(k))

## Publish the topic tree
Each branch of the tree is the data a different system consumes:

| Branch | Consumed by |
|---|---|
| `line1/fill/*`, `line1/motor/*`, `line1/state` | **SCADA / Historian** (fast sensor + machine state) |
| `line1/order/*` | **MES** (work-order execution, counts, progress) |
| `line1/quality/*` | **QMS** (results, reject reasons) |
| `line1/kpi/*` | **ERP / Ops dashboards** (OEE, throughput) |
| `line1/telemetry` | one JSON payload an IIoT platform routes to all of the above |

Run the cell, then watch the tree fill in MQTT Explorer. It runs for `DURATION_S`; re-run for more.

In [ ]:
def pub(sub, val, retain=True):
    client.publish(f"{ROOT}/line1/{sub}", str(val), qos=0, retain=retain)

DURATION_S = 180     # 3 minutes; re-run this cell to continue
print(f"Publishing to {ROOT}/line1/#  for {DURATION_S}s — watch MQTT Explorer.")
start, k = time.time(), 0
try:
    while time.time() - start < DURATION_S:
        k += 1
        m, s = tick(k), STATE
        # SCADA / historian — fast sensor + machine state
        pub("fill/weight_g", m["weight"]); pub("fill/temperature_c", m["temp"])
        pub("motor/vibration_mm_s", m["vib"]); pub("state", "RUNNING")
        # MES — work order execution & counts
        pub("order/id", s["order"]); pub("order/product", s["product"])
        pub("order/target", s["target"]); pub("order/produced", s["produced"])
        pub("order/progress_pct", round(100 * s["produced"] / s["target"], 1))
        # QMS — quality results
        if m["result"]:
            pub("quality/last_result", m["result"])
            if m["reason"]:
                pub("quality/last_reject_reason", m["reason"])
        pub("quality/reject_count", s["reject"])
        # ERP / dashboards — KPIs
        pub("kpi/availability", m["availability"]); pub("kpi/performance", m["performance"])
        pub("kpi/quality", m["quality"]); pub("kpi/oee", m["oee"])
        # one consolidated JSON payload
        pub("telemetry", json.dumps({"order": s["order"], "produced": s["produced"],
            "weight_g": m["weight"], "temp_c": m["temp"], "oee": m["oee"],
            "last_result": m["result"]}), retain=False)
        if k % 5 == 0:
            print(f"t={k:3d}s  produced={s['produced']:3d}/{s['target']}  "
                  f"good={s['good']}  reject={s['reject']}  OEE={m['oee']:.2f}")
        time.sleep(1)
except KeyboardInterrupt:
    print("stopped.")
print("done — re-run to continue, or run the disconnect cell below.")

## Stop & disconnect
Run this when you are finished (frees the connection).

In [ ]:
client.loop_stop()
client.disconnect()
print("disconnected.")

## How MES / ERP / QMS absorb this data
- **SCADA / Historian** subscribes to `line1/fill/*`, `line1/motor/*`, `line1/state` — high-frequency sensor and machine data for real-time monitoring and trends.
- **MES** subscribes to `line1/order/*` — what work order is running, how many parts made, progress and genealogy.
- **QMS** subscribes to `line1/quality/*` — pass/fail results and reject reasons for quality holds and SPC.
- **ERP / Ops dashboards** take order completion + `line1/kpi/*` — the business view: order status, OEE, throughput.
- The single **`telemetry`** JSON shows how a modern IIoT platform ingests one structured payload and routes each field to the right system.

## Debrief
- Which topics update fastest, and which system actually needs that cadence?
- Why publish both granular topics **and** a consolidated JSON payload?
- MQTT is lightweight pub/sub with no built-in structure — the *topic naming* carries the meaning. Next: an **OPC UA** version of this same line, where the structure is formal. Compare the two.